# Cookie Cats A/B Testi - 5. Sonuçlar ve Karar

In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

df = pd.read_csv("../data/cookie_cats_clean.csv")
a = df[df["version"] == "gate_30"]
b = df[df["version"] == "gate_40"]

rows = []
for col in ["retention_7", "retention_1"]:
    success, n = [b[col].sum(), a[col].sum()], [len(b), len(a)]
    _, p = proportions_ztest(success, n)
    low, high = confint_proportions_2indep(success[0], n[0], success[1], n[1], method="newcomb")
    rows.append([col, a[col].mean(), b[col].mean(), (b[col].mean() - a[col].mean()) * 100, low * 100, high * 100, p])

pd.DataFrame(rows, columns=["metrik", "gate_30", "gate_40", "fark_puan", "ci_alt", "ci_ust", "p"]).round(4)

,metrik,gate_30,gate_40,fark_puan,ci_alt,ci_ust,p
0,retention_7,0.1902,0.1820,-0.8183,-1.3264,-0.3103,0.0016
1,retention_1,0.4482,0.4423,-0.5915,-1.2402,0.0572,0.0739


## Sonuçlar
- **retention_7 (ana metrik):** %19.02 → %18.20, fark -0.82 puan (göreli -%4.3), p = 0.0016. Anlamlı, bootstrap ve Holm düzeltmesi de destekliyor. Deneyin bu fark için gücü yeterli (~%88).
- **retention_1:** %44.82 → %44.23, p = 0.074. Anlamlı değil, ama deney bu büyüklükte bir farkı yakalayacak kadar büyük değildi.
- **Round sayısı:** pratikte fark yok (medyan 17'ye 16).

## Karar
**Kapıyı level 30'da tutmayı öneriyorum.** Kapıyı 40'a taşımak 7. gün retention'ı yaklaşık %4 düşürüyor. Test grubunda bu, gate_40'ta yaklaşık 370 oyuncunun daha az geri dönmesi demek. Günde 100 bin yeni oyuncu gelen bir oyun düşünürsek her gün yüzlerce oyuncu kaybı olur (güven aralığına göre 310-1.330). 7. günde kaybedilen oyuncu sonraki haftalarda da yok, bu da zamanla aktif oyuncu sayısını ve geliri etkiler. Kapı zaten 30'da olduğu için bu kararın ek bir maliyeti yok.

## Sınırlamalar
- **Gelir verisi yok:** gate_40 retention'ı düşürse de daha çok satın almaya yol açıyor olabilir, bu veriyle ölçemiyorum.
- **Gruplar arası dengesizlik (SRM):** nedeni bilinmiyor, sonuçlara bu yüzden biraz temkinli bakmak gerekiyor.
- **Level verisi yok:** kimin kapıya ulaştığını doğrudan göremiyorum.

## Sonraki adımlar
- Oyuncuların medyanı 14 günde sadece 16 round. Yani çoğu oyuncu kapıyı görmeden oyunu bırakıyor, asıl kayıp kapıdan önce oluyor. gate_30'un daha iyi çalışmasının bir sebebi, erken gelen molanın oyuncunun oyundan sıkılmasını geciktirmesi olabilir. Bunu test etmek için level 10-15 civarına ödeme olmadan kısa bir mola ekleyip erken dönem retention'ı ölçmek istiyorum.
- Gelir önemliyse kapı testi gelir (veya LTV) ana metrik, retention kontrol metriği olacak şekilde, level verisiyle ve önceden hesaplanmış örneklem büyüklüğüyle tekrarlanmalı.